In [ ]:
!pip install -q youtube-transcript-api langchain langchain-community langchain-google-genai \
               faiss-cpu tiktoken python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Step:1 Indexing

In [ ]:
video_id = 'Gfr50f6ZBvo'

try:
  ytt_api=YouTubeTranscriptApi()
  transcript_list = ytt_api.fetch(video_id, languages=["en"])

  transcript = " ".join(chunk.text for chunk in transcript_list)
  print(transcript)

except TranscriptsDisabled as e:
  print("Transcript is disabled")

In [ ]:
transcript_list

## Text Splitting

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.create_documents([transcript])

In [ ]:
len(docs)

In [ ]:
docs[0]

## Embed & store in vector store

In [ ]:
!pip install langchain_huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
vector_store = FAISS.from_documents(documents=docs, embedding=embeddings)

In [ ]:
vector_store.index_to_docstore_id

## Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke("what is deepmind")

## Augmentation

In [ ]:
!pip install langchain_groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
model = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    answer only from provided transcript context.
    if context is insufficient, just say you don't know.

    {context}

    question: {question}
    """,
    input_variables=["context", "question"]
)

In [ ]:
question="is topic of alliens discussed in this video? if yes then what is discussed?"
retr_docs = retriever.invoke(question)

In [ ]:
retr_docs

In [ ]:
context_text="\n\n".join(doc.page_content for doc in retr_docs)

In [ ]:
context_text

In [ ]:
final_prompt = prompt.invoke({"context":context_text, "question":question})

In [ ]:
final_prompt

## Generation

In [ ]:
result = model.invoke(final_prompt)

In [ ]:
print(result.content)

## Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(docs):
  context_text="\n\n".join(doc.page_content for doc in docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke("who is demis")

In [ ]:
parser = StrOutputParser()

In [ ]:
chain = parallel_chain | prompt | model | parser

In [ ]:
chain.invoke("is topic of alliens discussed in this video? if yes then what is discussed?")